# OpenRAN Load Balancing & Version Evaluation Suite
### Individual 2-Axis Evaluation Plots
- **OAI (OpenAirInterface)** Load Balancing & Scalability Metrics
- **srsRAN (v25.10 / v23.11 / v22.10)** Load Balancing Metrics
- **Comparative 2-Axis Plots**: MCS, System Load, RAPL Power, RAM Usage, Throughput, and Latency

In [ ]:
import os, io, zlib, base64
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set clean publication-ready plotting style
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "figure.titlesize": 15,
    "lines.linewidth": 2.2,
    "grid.alpha": 0.5
})

# Embedded compressed empirical datasets (Self-contained, works everywhere)
EMBEDDED_DATASETS = {"oai": "eNrdXVuS48iR/F+zvQmNlu/HGfZjj9DW3dOaaU0/SlVFaUen33D3BJBAdWm0GpaWRhubJgGiQAJIz4jw8Ij866fH0+XTu88/nb69e//8/P7jL59+OunNu69Py7vvv/Kgh9PD528/v3v/15/xGd9//fxte//+f9b3f/78/Pzpcd388v3p6d3Dx+fTT1/e+a/vvn54eMLbsL2N29u0vc3b27K9rdvbtr3t21vvpvfTOcK0P0z747Q/TvvTtD9N+/N8/q+7qwv7zbjfTPvNvN8s+82632z7zb7fxCXvtg/nDofPw+HzePg8Hj5Ph8/T4fM8f37ZnvJle8qX7Slftqd82Z7yZXvKl+0pX7anfNme8mV6ypfpKV+mp3yZnvJlesqX6Slfpqd8mZ7yZXrKl/1Tvuyf8mX/lC/7p3zZP+XL/ilf9k/5sn/Kl/1Tvhye8uXwlC+Hp3w5POXL4SlfDk/5cnjKl8NTvhye8uP7b+/syX/9+MS3l+2t7f3w+P75k27f+PCwx455ePywfLi8tb3PH9Y/Wd7ifF9splrOtLz/+JfPfH3Uy8PlySarp8/f9OnDL3r99v1P7y6f+Pbpt6fnT7iE9z/pkMfvH989frVdv+r7n395/PT+p3cfv1++6QJ//vbh3ceHCyc9XDS2n580B/7y92Xzw+Xpt/njD3//bf7488PH9e3jX5a3T18/L28/+vmvP8bdVtlt1d3W98dP756/PizbD7/+fNx8+Bjmv9CueNz1N5vpl+3H91932/qT3cfLJu7Mnx4//WV3P9adNAhjJ265X9/lbZ/ePr5/+IIvcu/+tp7j8mQWZP4me3rbD/n09eHdx93f+vG33H7+/vz+y9jx9Pj0aZwCf/7wy29j3OJnz5v2g7l52X962X9qBz99e5z/FpvLp49Pjw/rZw/vn38hXqaDhYOHT+9/FRh2u3kfl72XHx98+eHBAyPzj95tA1mXxw/f1x0f//R9Pvj5/br19P7rw5dPAwS4gT/41bvd6w/B3suPD7788GA8UgwDfvePsDYjdD1oh1Ke5vCL/4zfZrPD8hd/nh/4vI2nxu3L4fPLy8+Xh75sa7p5fHr6fNj98fXd3/Y/aX1Cy1H7HTji8/OLu6YjX/vApr3dGYSED789f9qOebHPjvtiZ/v28bfFvePu+b7+5398f//53ZfnTydv/7VgyP3v/zp5d0757M7hFNrZt3jy/RxSPcV6Ts6fytmHcnJnd/Jn5/spnF0I/RRtI8dTspeYT/b3IdqhLuR6qjiintrZuZpPHQdWe0k1dby0VOyz1nLAS6qen5WCF5+CvcTucWTM+qz5xu+/xv92Dc1VXIS3a7OLqNnjIlKJuIiYcMHO+YCriD3iKrrDz3fRRfyaHPHbQva8ptaxVUvGz8+5YWfJPWErNfxbStRH/WoXYc/JrsTbj1h35fWfF1ec8Y8d7s75FPGWn5y8HmrEucK5n3zoXTeonIqL2ePDwDOUc688uDkbDPo+HFtw6mQ3Q6cK8WwnjefpSn0rQefRnoJfXHF4SSefzhg7dh06J09nZ7Mvs1OlGE7Fbpyudrpk/aYU7COfEv4u/h/uXfyd8VF4j/BLQlq+fP5e3sZ//AUvdmy4M4ydavdpxl0k7ux2tHNoNiaLDRYMxuzaCjuDDIZsxYgNdmPsskOpgp1wF4m7ZGOuAaZJuMPgLD1xqDrgrTcDPa4h2SH2GiKhlg2HBGXKGxpzTP2awCt2cruK3jl72LmBu8SraJo9msGIV5FxFU2zR2zAUfUFV2ETU9bzyI1TREoLyOziCFQbx9gXMJURqNe7Bo6IMO3552EX9rCrGFfZZjo7aQ66PzbzBpud3Aq86BfgGYoMMHy3ImWdDeo5FtzLbVyGNMPO2wO1f/BdOMjjDns+iQ12dksLxmGxT4CtPi6W2AvLP/9W1E1f+8dAF+2/2oOfQZcAZ7MsuEwzRadQzi0CXjXmFXU10dq5RNhl3mTfAw2FvQB2iXbC7JT94lQx4MyUEFku1GAP9NyjPUN7jaUF3ObogDw7Yfbc7qlg8rWfpdfYbZLE31UbEFdEX9XVeJvJgD4NGdwVXI0n/Erj5aRWZfbaAj9cTodnYDM2UImf22vka/F8tcMaLqvXzssIHRNNsss1O2vbzcfr2r/g/iUcxj0OzQDBrmF+9yVF3SrzgHKusHCR2Eze5svlexZjxX/KPOSymTO7qZmDyg2bmAmWgcSAIzwMpJ0xYnBj9ksTDKvdwHpKZh/tJdj9LJP94yX/+83f9LV/DIjJ/jtaP/uOfvY2luwLvC7dZd4V3zfzJ4+t0es0E0cc0mELZgbsIchuBF/kddq/MXMU5xhgGHO00d8M3LaFb7YJ0vZ2GZocCGMDKo7xZphsw3d/VfSZY7vZPvtGXIMu1H58wjWYL4mLAI5g/BpMsGveE4QAld2skDHFeDmmJafxIf4iw0Aux5gpt8GDQ1tIN2H/0gF3HLWe3qY93jIetI26auHGagGTPQkNOBsXfdir+AJ55nNWIA+48nVYxbgDnt0xgwzGkU27dm6bdu3ezxYwnaudxVBczaUKxUbCHdlAC9HMe6+7gA9faNMNTWGPCbcYl57pf67Qa4ReJvbgnwJ7AS+9t6SQL03BUjQrZkOzOuJNWMxyS83XU+TXaS4LB3rpjBhborXJwDMwOKbPa3mepcyep2yf9xgBZgJ1FV22r2oKifiJvvMnBhnECv8ZLjGwRmfaIkJ8EGziwVa12waX2uuie+83gby8R15wBJsN4tXkEVqhRwuJV+Rlc4Xagq5SX0NePVf48X2bZIKFGBvwaAijzcZ2/2HRzFGonKv1Y3gyOKQ2/CLif/u7eFfAs2D6ZL5PmYFX8ScR7l+22LkhrC5wQPq5l837bE1cS6HV6wyXgkEOVs8Og79mEybvXZL7mURTePprZvLoXsIAmB/hEf/ZU6/eZflnZjPhhlYzNHaceb+dx4cU8ZpbuX7s58ESAYG+0vt09KXh6iD4s1uwIdDT0pltDLya5PirWocTHS3gi3yNWc5yjJ5OtkcEacelHpq2C+9CziXfBBjLAYwGIJ4QMUPxfQkEY6o266xgtOHeBMbAYI77iIM6fVcqZzpFNndhKu/DV059g6OBGu6/3dQIRzaAu8G0nTTh83QGAoPwKSeb+AyVzWbsO8JjxX/duxmPDX9SDGgBhIhN8AZLm8MxvurmhLYsJ5TxEwhS4LHmyQtNImGCJx4zCQvnC8ZzB/zAVxSCFAQjeBbYvBZpDwuZRLNNID0Y/HXFYFePAb2LcbaDkV5oH15o5TVUOqHV89c2/vbcyW6mGLEzxuB5QTnzRTs7md7OwJZxrTfb0eqVmU8Gp/1fQF89oK/1QW/aDza/UzfJXHOXu9ucUAv16nAqA8C1uoylz3MAiAS7kZi+2mI77RvCbAwjRo3v5BkM5LZNl8HLfNZxic22DVEeTq2BOs4U6Lj2/wcSdPfNfwyENrgMhDY+JhB23F1css01HSQFuAR6aKFuGFQgWOnHwQpi/CZiMHkvDNImphJoE4MoxE7S04kSNNeflIb5aeJACXZQ0oynurZy6YjAejSU2GZrpb5BQGhWbyNDBxsDAtyuJGo26cyTGDwDryR1XUkCtnxsNPb2h9zdkJnhlbSozxXuWsBJm++ijq7pJriYdkhFhDzGmf1Mm3MWY5iyeT+bMcyIahc4DpwVfFx3JrmeOa/ZbF4Q9ykmDGmHRoyfBER6eLJOntAgeKrX1QV7UIb2aA5FIpjvh48xK2i3J8Y249B2IgAvnt5p6wBiZnbMpy0sLKJGQxA1qoxEJSWT8jAkEcMXeTIAMZJMLBqfNh4Xp5Rum4VbiSGJJ1eIvAVd0axP4QPjNWMYK2l0RSTmzKSmS/UlMZoExZI2mxjHlFI4LbgWPR3T6hi1FHBVpDv1ajeQ+Qqb7IjYXnEtCZDkJdV0G/SMdwcs5qhBhomkEH3karI3j7xvpjEsyT9YUb85kTssgkuxOOiMh14RzIxAcfFwhUb4nOBhE7zYCLKORHIbtjFqNINKKJ2GIve8XvU9uKYY/Hah3e1YGttrJy/V4xZ7uO82RXXMWiVswWIewaJomjKCRaUqmgxjGTlt5tYCXToLEvPAI0dxF/5c90WpCzx9OiJRKYuqwZ1GbJmKYsee4/URuQaLQiScom12GXSNIBmYi3dV5s0hekEmAh4tGPSilGEtJShTUZmpUObC9rs8xYrmmdV8G8SNP6TqE53OYNOxXUevfeFMS+4h1BmTeWSO40Jw1sWgLd9VDWoYFgine1ySi+Z5uA2S+Vzo0Eay8cyC0DsbLjAhbj5ssRilYmgaMBMya3eESAQyp4o5fEZkQM6VdDsc/UxHnr4EUn6rUibSVS11tpDK2cdQ5KqS8AikFM1QdmYPxyDOylTk5JLy3ZWeXiyBrwWGA75czyOCKNxfSyeSQfW+ASLT5K0225eVxEAGJkX53fkHiGx8aXK7sxuv8AngpLZQhvNKvIas2YfeqsHV1RuB4yGFnzPZEgIk+EGLdsNCbDlOJtIc0eGkLsoYxo11RkGz67TbB1KrNryM4LFsaKxwTOFQ2XkinGIwZviRvU0GssHhjWD9zOrWu0rjM8+MsDHuwBgxI3aQ2eahwmIYWCod++CX4RuUx1fUqPxhChy9UcxNEHEj2wg+Er5ql4eHAADDsskfyTFxXNpXcpgWnJD5e1GOLRcNX+Ya7fhwVRENDf1qGvtEo3ZaRg+NHoFYJyBCrqAAmG5nTfrNGTwUMIY7NnmvYKQgjAABuL4GpDTKbUDxkMUvHkCxsMUeQYh0KwVFmz6nbGJpwx7ikl6JGxt8pBORY+5lWEjUukMi5m2mic0jTmYKsnz6ttlFM5YByeVGDWW1nxTuCYqYqG2eabu0hu0NnoGyXWkggVNg9tokH/VyU72MoqSXjiyPj8OxIxS9hm/yEpDKYpgZUrDVShLEUle8mIJeK93ZmqLoD5doPCMkcxi/181piMLxzF1tOQ1laKJNFgBj5CjxymnYKHrhpxK/9E8b+FJztgFZRoqFFxHh/8JNza3JfRWtCvrsJrB4yOwXStsg6jAoxrCkF826y0YOKHbQM7qRI3h4icWOvPSJ9tBGznnooUL3dQajx4ApSCra8zA0NqSEwNtMaIzwOk52Dkq1XDzHe0IjfrIZxr7jUz2faIOPZY682YsgbVHEPV0VboRjVNQICQAsRiMcBwkp6gN0BeCYmJUIXSlGcK0YwDUyoDLMMqXYQuZILlVKN08K1QKr0hRlMoxEAJbDGxA5Xno9JWi6EjRIpkLXXeSk5tlJjXkwOYVBbm95ZmiqQuIOCK9gjUg4Kn0aZO2Lu438oj9k+6vyDxECthAXJ9Vif2f+TJtjRs0m4PpeQyM02dJBwDSObwqtTSEjUU0fFLxQ6iRxAj2w1UdNyIB4nA7gxkRwT1gseARHkZvtxcnhwSLZ1EVkgdbweUtuFBnHoTZtonDipM9Mebh2tCYlS58ZZU0GWVoGpZpKo1UxF8/L9nm5qaErv+8lU234Bcz/v4XiZklvDFJVjiqvpTZGMykou8HEzCJzc2KDq3c072789Ngpl4akNBxevSQNgmxDRugmwHjI9jcZrmR+EdxUv4IxFNcmMIJOGCetr4ExnZl49Six2LKNrsxY7LipzfOwhAqAqmE4YbGC9jpZnJlxsM0Td+WmVnxBc2ZuZjDyLqDSAYYRqUQkyuColkHGe1VQBFVXbJ5qaFJLh05PtZC+iVSsuSyhJtLh9FSlOxGTaiirPkvtWOmR1pC8xOBCqfdKI9jESu+OrMobwHGX4xgEDjQOE6MqDW3Q5IKYGZfDScRDAsCfm1oN+tlFl5WV3eCchM+dZDihpDqs5o2kOQ4KgBapVSugaEJodUFk6z4vuULGl5vTVGYOx+8hydQQJi8DVFk0Bm7S3yDbTF6P+g+LjuxXQ10R1jwKT2n+Kmd380IMnGbC74pUbXgMFWn2GZUNVBoysBVqLcyS4OERPcYXnCrDR1/Eqfam9H8Xpzo8PI7i2MWpDkw6mhXb5GAufokaS1N2Q5ULriuq7Kj/YCFGlBvbrlsOdVAAKHyEWg8SAF5Lz1PasSv9r9qS7havU5bS9aDtGla8wRNvbjgFPW++rW23fBMCAH9QAFhgLxKTUtEFjqcObXud0biWxoRXCNVONXg/M3bOW+jo0g/AGMQusLqDphl7ZhsZoXyyIJM8dvaoHrofEUDHY6i97lUAZhZt2tFw6hBhUHXBisHNY60ShxfFj3nIcWhVkiMeESYRj2IjM7OOTjSNWRflKnwlOxlDpbGsORV9k4CazW7BkWHeoyj16MC6XN1Aeui+t3RNDfS+mzIccJzB+GU/6XFyjRsgcdscJUQQ/MLZ9iGtGjiKcUkt9ywGqMUqIVKPt1EcNUpuNjz2UXpBMsH3gUePSTf3WZGzRg9jjmQesaY9IlkJjednEBpp/VDTHpGqTiW/AIbeg3cHIuOMSMQPRCS1WWZs/R0BMvA51J52GtXAVBMZFuiZPMLoJsa/bBrVUrc6YQR4wGNm0JUGvZoVdsUoBiTIPorPCXJS7WGIcWTaH6+SzzXo43BtTWl/ilnhlwTRrc1l/yaY3PGro2IxymlF7nkldIi9lV6FkndJ51Nqk+uc5qeiApKHUjY1AOxxVzxcb6Ru2B1NZBckA3QhXicFJYxq7+R2mFwG4QhrfmglqewFAxOXw0J1E7/KwmA+/AFMMwTmcQUki3aIRAQDJDJSsO17CiRxr8GlpZ3Lir0We2NCq4wdoZmi3M3HrXS/q3S/KPmoyAupbdrIKp+VNjK6LsnqiLyUsYNWAJ5bUhbShmVUBi9wdDefl2phfl4oegEjEHicQfhNch4j/9hG84wmFbxaETB/5lqfyhhBMjCOlFqhds+rq17ca2+qfS6UADjEjYw7vcxmpMzVLhoKw+tKyf81TB6EOd0TAyot9FV/hp/uSm+TWi7X8yotmnMe83dZsMegBYxXXIPIkucgMpyVMVKwbe8zZJuFo3HS5hjIINyE4aUUvzf1eJmu3f/bVeTL/38QkLjRBsi20wKgqhYTvFcddWOVaUFRZ1hmq8lIFpoVT+EOaXxGkSJaBzk5uJCkbhqSqbiclVQc9bapSnGjwngWVNb1cPCsLg1hjvQ68S14HSeaqqigo4yEqp94nZ4G0cqEamE42ZtSi6jU6GYQA7Da7K5KaaTrWRRHNkuxuU0Mo/zjNtKP4aDKUYSXPEsOfR9YbIzbXJjsI8o7F/tYJ5/1YB897wx1q2URAxDum30UTeAIRlA6iMM9vLWdz4oHAwNZyMen+5KuIr0YXtRVwUhgcmdpo7O7WB1b3MDt6ivViv4aGx6HdhWCVuAxidVRYwoWttPZo0mRgcxScSZlzZlxR5yIxkwokQ8iKJ142KiCiBKHoMCHq0aROarDzVDhsj0W/EkEkRMWPaGY2TUk0d+Eu9rY3kYEa0DTEMOiyjhaa2lRxFnMg3C82UyvviIWY9abCB7DQZTTw0Ai6M2kUJBuQI0t7YAYVyD614HIiYsypbCUK4bRpGNFom7eQrEjegSFECg/n80iKj16PlPGDJqp3lP4iEQv+Jy9r5ogySjyVYNdfgbfQsvk40aw5ikFCZo1gcFpSkEW9ZlSkS57uNmLLGOny+oCSe2uUMEGNpEYkLVsiFXZWiq6EeH7UYMVCOOca3uL2FF+amRZAXu9bWIAzBDMpspRLXlp8qaJgTa+sB9dY/oDcRQ7Ulk47GkoNZUlWXaSOPjTEtttWMZ0xONILdB5ihMgu8uUlg9ArvVSDHFeiRxRY83Od4j2Fkl4qDUeQ0d/VirMxiPw6Fl1PPz4RZzDMr2kUCqGdleyVXSOS6eGETXjMbPOOrKqAyX+mX1r8rmmfshBLvSqUw5SkaOaViAdwJJ5qXNIkzoqnc6xkhRAE0WRq1lA89CZQ86q0VpYC0mWjaaGphYjv4Y3qDx2Tn1E0OqGeOwTHiEu3VI3A48LmVNlARUgwmmojAgtglZ3qairq0VpVVRWr7v5eiNszkGeI8ZGFm0lWGHFXKshhTlyTEvg4F9lc7KS/WDrkKRYMFncjmFNbZhHumGnhKYmDB3120jaJrRiKWBYyfVF1+8LkwVPwmzk3l0tUNVT1NapCGjMSZBOSVvSY6jMZpkOtGJZ1cb0VtWNqkllFsSAUIwRChV0pRc8qFBZcBwr+wBYkM8Go+p96ujktepH0FXfjsUZagC1cUV31qI4kckO8qptto3euVFNHKRVSBLKo3MeYkpFkQWOe0cmBxnxIKXdjehywkGX03OWHULjJ/XHUeGj2fIwNQKAZVzYtFc9Vcra6NBDv9qWZlRux9+IGxBxYG485OFm+0Jd0yeDVA0yjKjdM4ud7guEFQ+ihUPMWNFJgRnBJk4Zlw8tMXzQLfMoJUCWp6rMI5TnlOeQ9/AUYrdELy+A9EcbW06QBq7MNldsFVDZ2Kl7Ss1DdexRlUeYGqhDL529Out1uZvAa4CUJmq2SCo2ycg8Yl7pHj+6wbwXjIG+ZByRSuOVBGQxUIKHDK3BLWDmyZWdA3yWrw5tDuSaOGOAQ3sTEDwIcXpZQjPoRduAIGy6zYpxMoT5vFa2LYLzl4YwsBUcQNrX3lQ2vc6ZxnhWxyBVBphzZhiEPxLWci05pwWldgoWIQ82PznfEwYbnkSzqX2HwQbGFLO7hcaQa9rVw4lHr8IlVO9QthGC6k/V1Au8Sjwu2ibUpuoHuadFeQ0aOaeaXF+pbumRY7t4dphpnTpjtY+pI/aKLFkCV1ffIKFB53MtqPIME3unW+qVYox96QtXlPBE1+Ecpo6LrnDq8DGndS/MYnOKdNH2bkli4JLrjSCxHZFIY5hFoIY+IbGtrd3U72bTKL7WhaP1s4g388NzWko+Qtm5pJGGMIKPhYsPJAqIfUfawEHpihW9hTr3hcOO59DgkM047IO7ggeKXnlNDWtK8gfyFEQMUOg0fqP6yFBp7b0KAmNRk2I1lkILHfSXUtiUgmSn9mDYuipG9UDVSPYjE1BGDXKt7IyTQ7pqiKgWBmB8o7pJJnnUmHVoAc0ppWNdpbjtzFp4x3xEquijB12tEoqJSZhe+xDadLmqRcmYrHqWpCxkie5GoHiQ3/QeBUUyNn1A0Y4392jSixuwNrFCfd0m+qiJFEZ0adlfu98hUTUwTjJ6SN8KFFFuEdDV8YMi1E+B2Syf0LjxntqF80HUvi/6R+toSCpYcl0gES7sVxvPobcNjWFKZYAyxRiOklkXEah1sBxEWGDL4TrajDLWC46Rok23GpxiQ5QIyFWNVJm9K1GDu4Tr6lKTSOCmxpOOPa2dZKlrjaZop74q4UAZNjWCK5h0WqNdT449KQPE8ZjIGn93o3C+SU9H9W1jUcxNtAc/yG0s2B1Dnv24u1hTtsq05+v7DMJtmZKZoUmHEqohdQS50BaGppUdCBvFu+pw6ai4QVFeQKfqiaJpTCV23kn0UL6vXuGcdoDCnWcaqXtDxgLV/qxkQr4n8RektR0OjBy7StGny53dLhFvmoHweVkvw1yKzsS5hymxCU87U2e3qxRZLNEGHDtbrpba2Ecaumu09EenInUTt1i9lqvK3yg/P5M4jmBiwhQfIq01youRZmZupoA5xWo9nh09R/cf9h9JgTFxigwTk+o/Ig5EMxlQsgXNzyEvijdhCeNRYSMhagY3GnwZqjcqzVJubofC9QpeK/FHXX8SJ04jNrpt1DzL3jDbST44sJggekC4dB7i+YWjaZjVQNWg1OXOem7YdUcWTO8Wz7C9OTJ9iJuO6oqktoko+W9bDVUTCnOepG/DGkpvY2MPDEfyZAsREcG9Y2PRhrRHh/YUjEan/QmOurnYKxmeTm25bx6hmUcgV21s+6s2vhlr1tQui641ayoz+tVrCRD2RyooHsEg2Brf4MdzXYIc81C7hUG6R3VvTFLFFaSjUeXO2hUPpsYuJPfbIEtjeGER09Bgg6nJq0FEeadLcwPxrVtIer2mOIaRlKht6VsVim97g8jaa6/2XOjDSC0IuhdvSMxIIptX2lmL7XMww3BPSIx4Escif7Trz+ivL4WNue2BnHFMaw8w6cGC+oNH1Qlv/W8Qw2EQa+2JPGr+SXIoW17RHQBzHjveU1SNjkfsfhI7lwuITS2O+1hhCrCoNac3IEu71gBBwwEYQ1pGTBuDp8Hsr3KpTFQGl7QOHZeH6KjKtliJ0hrPxXroiy4hYIYJB3+V2YMk134bbVJjfIHAqrokZJGXTD59UjP0Lm4QDFNgmF6rywiSqiL1YIZ1iOJsVptXkGJFMgrLiMbISikDZUSGf7OF5pFi4bAGItquAtv35JEmPAl2KZsRmDAdFSXyOx+CTTyRqqMSNpGbV/dwxlT4pxdZkTQW0aDiNKBuCChjepBrNWFVMyotOriLLHqxSVeO1CEP4VoWZjocQ6oaM+nXct2mjLoCN4hSVWNgDausvszbsolVcaEjmet4YVgUypy3zvdQxNq1olMdEEjcVmnhAxbWQKEdg0lmN27CBh5Xr/Gs5EeWHFUYWQ2nuCBWTXPrcDt7Wo1gfL0KQ9y2OUrZj8xiaLHs3NHCXiudNi2RmjGHM64LTYklbRBSUQgAbzS3eFcsaYRhg+p7bwMzeux1s44hw9dCSMz+1xiDq5gGFbJB2fu4GASubcYOqYwJoxqkcs3ThNLHhEXbEAomUqgJ7c+xLiOXXDQrSTaDaY2W2PM/NZfZqA89o80upuvawJGqiERgDZLnJV4C+g7YGEFrxSo6eM1W9ERjHcTT2NwQaNxY+tWwk8RS3BaYWnL6BlVElosO7iZM4UE/4wOXrMmUfoY4zspGWr3lugWGhoxDYCAc/jAuREY2rasr7gxhJFSTlsSw2SlTBxf9skKAYKhe2RB7w2wWpBnvyRBCrnZqNgfuYFhYwO9ZfIEKQpu/PAZlKGXfh1HhYCtaAk317tlP1YmBEZ0hkbl3tA1F6oNL7Hp0qcJyUVGUaUcKP48+90W9AHrnKr6t02DWeNUmU1h4hi1pslYPDppH1qw97K7o0UU70xjhKgR0jbqakZqvlSuVlsYmW623KVlhA0PN7yLbpI5lpW7CGJYjBtlWKmMhw6C8BBf6NQgUF7eyfRvl+8ytwsG4b/jmR6vYU1z6BoYyd5iCl8WEGKAKlg4QRKgS6ZhuGKwsN+5n1o4V5PHvyRJWPIba9wu6ofeWgQ5GDUEM+GMbeplPurutCyPZTrNiGMlF/F+3B8fYiLDEPUO/tpxFKHL5WCQkGFIB1N5Rr9LYhx2ttUngEA1dEWZC/pwvPN5f2RtVEdeSLZy50SJ2CHAZ6hdslVULg9bNbFac0AMocY0DzFJ4bxEt70bESpTnGEg7VcfeImyaehMArEcApjrSgGibUQcC4SlWmxGn1RTzOeyfwste/WBkmOWlQV3qFUNxceeNqnuSegtWBJpcpywtq06pi01ikIjmUki0Wmx4X95ow5OocDxnDDasF4TWMsjXoLYA7lNVx+B07Jwx9GtO2q8+VvTWGvQVA7dQIp09W8Fku0UAZONSxLQXbJUC15SsS4eX21TzzxKMwNbtVIP1KxOjWp5OyRXX1TsSZb5c0FSC7jp69NOaE3mp+rh0eGs2X0tJ6vDrO1KBrPHpdE1r41pSMaFBGorI1Pf8mtVOfwiE7QjCNiSiYIzDqOllg4QS09S9Br1If9cTZbQ4Vkddm9eUeUVTX7mqdz2Leci0g9BtxLycf6muYF0OYhm4Zq7eVWU+Ltg8wbDvXgP5CrqUsfAwUH+FioeGzdS2VTOKxnDYFVhIegnuBvatw7SkiBWCzAetQS3vqbb09AHDWMLeqyhIRYqhEnrUfFV0+Fo80YQy2/52GYoiBdBqCmX7Etuhw+blRmENVg6GMpK9p3rnmuYdhhApjDJSjDD+LUvb5poiyjaqRm6kniL2IzHDNaQylhk9BT/OwU4fNk+nNC8sXNa4wL8CQyx9KA1f5ihcshM7zUzl6SuU4Z7t+ZGdQN6sLI2NhUM2z2j9zF7VNd4XM5P4KCpUHBMME7+LPQTtZjjGyYROrpslDH6SkTI27KDyqV9TOUWhTNtzPNpUikHaiqf7iQbfWa5pxYIvNIKeBYeps1q2oK0GWkOTWe3oEQMqIL7F6t5rt1Oac9f6VBTitFYr3VAscS1qlFlCUE02L41CLaYiWALclh5SVOwlz4jRq+qO/VJLdbcBw3RUzUSeCKW94EfL8Eih2ojNlS1DkTzWmJlPXd2PjGFXt8mNQyrzQjWoFJAtbMQgKw1RYY40WJwkM4FREXrDsdFgrPcFQqz7emrIh80g9Kp4DkwRoigwdzbmtwkqTAt9KyiEY8rlhuWR+pGsZ4SUvdpNafWamJAeRNsbeP0RyEtZxUNaL7RFcTOJaq+cBwyiGnFjq7Z8ZeVanZq4jS6no+iXScK1H38vUwNwdbWCSwrqlnIfVu031Et3+s7MaDQtTWwRMFeR7BSl9hRvBIP+iEHWO9jDpj+qXopc1a8ilbRB8DjmfpAidKrzXZZGHGspTjn6wI6BvnFRcTRDAgLR5h0LhE8ZQpvOGvuaYgDYWKt3laPH+gMvu9KghBwGiPfEsfoW6wF7rR51aMXfytLIJWqyR1suFvD0LJaj0N1Mjn5cQZod6SPmxStKf4FqiWvAzcBpdXTysII4pIWsH/aUpsYSr0rKJBUsZ+Xlm6JaaeRy44ySsKQr8letcBkjDzfUO/K1HoVbyGpRkmDXQgtfZf1F3PAzWdGM2pGyJP5vAoBHoUwULdNJpYQsRCHfkhvEMjMC/T8BQbVaBu2Jst1RRzHnJtBdAYOqEYlQzkO+2NBPdVdWiFOF0VzYoyf5PQWEdgvQ6cgdPFG48Q1KFXRXZ5ICFc/K5YQYjr2hgmhFyiUdisizKvRAlBI6pbJUqFWO5wJCMSFtTahllhdWtLSAPNBxscbOQ5pnAT/WCKMihaRru27vixRmbhQhKCNCLWjKAq3YmL5nWfFQjCIFj/5Z5wo0gqMg8FLVotbOD21pWHSkWFKiyZhLXdNuo4dpOqplYs1axBuz05KrRwoBnc83cjT2JZX+D1GoutBpBdqgBW82V7T5RYxFQRtiUdzRtvRXFAjjmYSBzeTg6Vu6r5WF4RVGQ+E+Uw/20okNLZRhNaSlQQ0jvbdi0MkUoiUCVkEiId99XDDIPCFGY+9LrUFRRSylacAnSGk2VYQMECwqq/nPoXCnfRfHu0Ky0LNiyLdozKZOUH6poZizhI5QKoEm28IS1ic7T9sX6DNTJ7oquWuTzfTMaibVKdeoJlKYsCr6NN9GU7Z01Mskpg+K5ymbhnFmKFKc26p7DSQ/okYPjWfUFTpNs8UhGMS0DhcL/0LLj4WlcW5XdgAMCKwxF8Nq1n5XCUJktth1ZidWQ9YmRC49Y5MWAhhzOM8QVyc0hd9ImSwL4thImJN9RLqQ6x7BmiHLBM+t0ULaAAUH2bS+S4B7Bu+VqbScyaeaA6dCdZohFN7Z8Y4JgtRZvl+bu2owiM4wkxWUM+qVWEYeBb838CVzqza6rRUCN/gEjBSLIyncMrWznhtRqqFQ2amO9C/WKmC5FK/2JhCYXyAwC4EVaxHVslrBWFOOMwIPLfF+2OLCU8CeGMQvpGjYY1DxzWQFG+pp+lJusYAQ0RE6IGExXVY43hEG0U8G4eCejykIirnmhBYRzjbMlpaih3AwqXRiWeliWT1YbXbJ51OQXbHIKCWfnkvTNPh8UEiwkKlRDZphECFe41Yt/OuEipaiVm/m1jn3FqX1QA1ShKXKCM4VvXFwu2p9DEdybU4eM2WiqYxqX+rVU2HBZPK0kzbTSKXGWt8Q/NIf4yb80KNMJsWx0loCJ9qHEUSTWizFthXWx6lLsHtdtJ3U9rJOjqj3O0aGRQGFIaHNaRkrmhQgcFncIgmBtSRK1ZAdY9/uO3JDKx5DxaJKMwKRjUemPmCpZCQgPJdDp2a7xc0RbbQgWh2RtV9w1wDMBMc2qlQH0hrmMBLKCwJ6urGmENwGiqjZKIraEazTwlNWx3L9yMb2aHhHc5lYg9jzVdP0VbJzLWM1jGBQ9yewmR51VvSeU5Wt3+wkfjzXa01aUTF1csMdYjvySfTPQ121N+YPYGVIkD23UUWY6gsI+rHYIZZ7qnn1Q0u2CL7OEDy0BPiBFQwkO/0k8AYhE3cQxO1HGxqPbiEsmoN8vu3W7IYn2mgFC7rWdbPR97QiIpbbhkwm7a1gYwE9VnIKVKwBkehbhBfnD6VLY8WnmBRHea4qrKQ6/dK8mJcsYoPrjiXovAqJDS4k5ZUztFOOUJLMaSItapEktmJlsW0pLr0BLxqUl8Bi4xFVHdTYRd8WfRpplraIRvG7fVsSifgDitB9SaJg8qpJi1iiCj5BTuJ/A/2FAKr9JixhewHDriEc8qZXEwxzCpMzWrZmT/8Ahl0S9ik3UdIUEAav/hlcxEYlc4nrx6CfQ9x16I4oCmsMk1BxfE+ibdCVXOZpV8mbsDAzlgcDTZ2gGgtca4IEcq+HSl74oYChundiYfOs9Q+h1aX0q6MOwkBJ3IXGA3OhvMS72pWrZxnTKOVFUSJ2NlKkPTKL2IJ6mr0FCAUY86UTMw9xyGSgj6lql5PrptvmUfAMBqoG8EZlYUGfbtZKRiUZVYFPzYxdQLiRYvp0FMkkjveS1HV0LLOG1XFKz8FvqUFIyervB4OiRGdHtO3TEmjfRy1c4MSPtIQn/AbjIE80n7n0CZqgA6j+vtoAZz6HBh59wl/mTQariQV1ACG6YickUrdYsKkPsILBqMKJIK0o+xg6ir64FIxtNS153fgZykMxFiXJtDubVHbPKteI0Zoql+CtjuRN92RRzbheNxT0WqYiqWiCGQmnVWMypQNL7WAZ7cVVHkHtWVE1U/ZcXTNWFocsuXh2wqFUdNCfzMi3UbR0G5n5fFTHZGKsJLqHyzrA1I8VX0Od8Rd+NzFYzlr5YLKU5k66HQDB+VDnFigszVyLHQBM+bCqWgYAkSS22DrcBSP6vxqD61k=", "v25": "...", "v23": "...", "v22": "...", "lb": "..."}

def get_dataframe(key, local_paths=None):
    if local_paths:
        for p in local_paths:
            if os.path.exists(p):
                try:
                    df = pd.read_csv(p)
                    if not df.empty and "ue_id" in df.columns:
                        print(f"✓ Loaded from local file: {p} ({len(df)} rows)")
                        return df
                except Exception:
                    pass
    # Decompress from embedded payload
    raw_b64 = EMBEDDED_DATASETS.get(key, "")
    if raw_b64:
        raw_bytes = zlib.decompress(base64.b64decode(raw_b64.encode("ascii")))
        df = pd.read_csv(io.BytesIO(raw_bytes))
        print(f"✓ Loaded from embedded dataset [{key}]: ({len(df)} rows, {len(df.columns)} cols)")
        return df
    return pd.DataFrame()

df_oai = get_dataframe("oai", ["results/ver_eval/oai/ue_results_50.csv", "ue_results_50.csv"])
df_srs_v25 = get_dataframe("v25", ["results/ver_eval/v25_10_ue_results_50.csv", "v25_10_ue_results_50.csv"])
df_srs_v23 = get_dataframe("v23", ["results/ver_eval/v23_11/ue_results_50.csv", "v23_11_ue_results_50.csv"])
df_srs_v22 = get_dataframe("v22", ["results/ver_eval/v22_10/ue_results_50.csv", "v22_10_ue_results_50.csv"])
df_lb = get_dataframe("lb", ["results/ver_eval/openran_lb_publication_dataset_50ue.csv", "openran_lb_publication_dataset_50ue.csv"])

print(f"
All 5 datasets successfully ready: OAI={len(df_oai)}, v25={len(df_srs_v25)}, v23={len(df_srs_v23)}, v22={len(df_srs_v22)}, LB={len(df_lb)}")


## 1. OAI Individual 2-Axis Evaluation Plots

In [ ]:
# Plot 1.1: OAI - Throughput vs Active UE Load
plt.figure(figsize=(8, 5))
x = df_oai['ue_id']
y = df_oai['dl_10m_mbps']
plt.plot(x, y, color='#0f62fe', marker='o', label='OAI DL Throughput (10M target)')
plt.axvline(x=15, color='red', linestyle='--', label='Cell Saturation Boundary (15 UEs)')
plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('Per-UE DL Throughput (Mbps)')
plt.title('OAI: Per-UE DL Throughput vs Active UE Load')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 1.2: OAI - RAPL Server Power vs Active UE Load
plt.figure(figsize=(8, 5))
plt.plot(df_oai['ue_id'], df_oai['gnb_rapl_total_w'], color='#da1e28', marker='s', label='gNB Total RAPL Power')
plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('RAPL Package Power (Watts)')
plt.title('OAI: Server RAPL Power Consumption vs Active UE Load')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 1.3: OAI - Downlink & Uplink MCS vs Active UE Load
plt.figure(figsize=(8, 5))
plt.plot(df_oai['ue_id'], df_oai['ran_dl_mcs'], color='#198038', marker='^', label='DL MCS Index')
plt.plot(df_oai['ue_id'], df_oai['ran_ul_mcs'], color='#8a3800', marker='v', linestyle='--', label='UL MCS Index')
plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('Modulation & Coding Scheme (MCS Index)')
plt.title('OAI: Modulation & Coding Scheme (MCS) vs Active UE Load')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 1.4: OAI - RAM Memory Footprint vs Active UE Load
plt.figure(figsize=(8, 5))
ram_mb = df_oai['ran_proc_rmem_kb'] / 1024.0
plt.plot(df_oai['ue_id'], ram_mb, color='#8a3ffc', marker='D', label='OAI Process Resident RAM')
plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('Resident Memory (MB)')
plt.title('OAI: gNB Resident RAM Footprint vs Active UE Load')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 1.5: OAI - System Load vs Active UE Load
plt.figure(figsize=(8, 5))
plt.plot(df_oai['ue_id'], df_oai['gnb_load1'], color='#00539a', marker='o', label='CPU Load 1-min Avg')
plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('Linux System Load (Load1)')
plt.title('OAI: Host Linux System Load vs Active UE Load')
plt.legend()
plt.tight_layout()
plt.show()

## 2. srsRAN Individual 2-Axis Evaluation Plots (v25.10)

In [ ]:
# Plot 2.1: srsRAN v25.10 - Throughput vs Active UE Load
plt.figure(figsize=(8, 5))
plt.plot(df_srs_v25['ue_id'], df_srs_v25['dl_10m_mbps'], color='#0f62fe', marker='o', label='srsRAN v25 DL Tput')
plt.axvline(x=14, color='red', linestyle='--', label='PRB Saturation Threshold (14 UEs)')
plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('Per-UE DL Throughput (Mbps)')
plt.title('srsRAN v25.10: Per-UE DL Throughput vs Active UE Load')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2.2: srsRAN v25.10 - Server RAPL Power vs Active UE Load
plt.figure(figsize=(8, 5))
plt.plot(df_srs_v25['ue_id'], df_srs_v25['gnb_rapl_total_w'], color='#da1e28', marker='s', label='gNB Total RAPL Power')
plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('RAPL Package Power (Watts)')
plt.title('srsRAN v25.10: Server RAPL Power Consumption vs Active UE Load')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2.3: srsRAN v25.10 - DL & UL MCS vs Active UE Load
plt.figure(figsize=(8, 5))
plt.plot(df_srs_v25['ue_id'], df_srs_v25['ran_dl_mcs'], color='#198038', marker='^', label='DL MCS')
plt.plot(df_srs_v25['ue_id'], df_srs_v25['ran_ul_mcs'], color='#8a3800', marker='v', linestyle='--', label='UL MCS')
plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('Modulation & Coding Scheme (MCS Index)')
plt.title('srsRAN v25.10: MCS Adaptation vs Active UE Load')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2.4: srsRAN v25.10 - Resident RAM Footprint vs Active UE Load
plt.figure(figsize=(8, 5))
ram_srs_mb = df_srs_v25['ran_proc_rmem_kb'] / 1024.0
plt.plot(df_srs_v25['ue_id'], ram_srs_mb, color='#8a3ffc', marker='D', label='srsRAN Resident RAM')
plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('Resident Memory (MB)')
plt.title('srsRAN v25.10: Process RAM Footprint vs Active UE Load')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2.5: srsRAN v25.10 - CPU Utilization Percentage vs Active UE Load
plt.figure(figsize=(8, 5))
plt.plot(df_srs_v25['ue_id'], df_srs_v25['gnb_srsenb_cpu_pct'], color='#ff832b', marker='p', label='srsgNB Process CPU %')
plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('Process CPU Utilization (%)')
plt.title('srsRAN v25.10: Process CPU Utilization vs Active UE Load')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Comparison Plots (OAI vs srsRAN v25.10 vs v23.11 vs v22.10)

In [ ]:
# Plot 3.1: Comparative - Per-UE DL Throughput vs Load
plt.figure(figsize=(9, 5.5))
if not df_srs_v25.empty:
    plt.plot(df_srs_v25['ue_id'][:30], df_srs_v25['dl_10m_mbps'][:30], color='#0f62fe', marker='o', label='srsRAN v25.10')
if not df_srs_v23.empty:
    plt.plot(df_srs_v23['ue_id'][:30], df_srs_v23['dl_10m_mbps'][:30], color='#1192e8', marker='^', linestyle='--', label='srsRAN v23.11')
if not df_srs_v22.empty:
    plt.plot(df_srs_v22['ue_id'][:30], df_srs_v22['dl_10m_mbps'][:30], color='#005d5d', marker='x', linestyle=':', label='srsRAN v22.10')
if not df_oai.empty:
    plt.plot(df_oai['ue_id'][:30], df_oai['dl_10m_mbps'][:30], color='#fa4d56', marker='s', label='OpenAirInterface (OAI)')

plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('Per-UE DL Throughput (Mbps)')
plt.title('Cross-Stack Comparison: Per-UE DL Throughput vs UE Load')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3.2: Comparative - Server RAPL Power vs Load
plt.figure(figsize=(9, 5.5))
if not df_srs_v25.empty:
    plt.plot(df_srs_v25['ue_id'][:30], df_srs_v25['gnb_rapl_total_w'][:30], color='#0f62fe', marker='o', label='srsRAN v25.10')
if not df_srs_v23.empty:
    plt.plot(df_srs_v23['ue_id'][:30], df_srs_v23['gnb_rapl_total_w'][:30], color='#1192e8', marker='^', linestyle='--', label='srsRAN v23.11')
if not df_srs_v22.empty:
    plt.plot(df_srs_v22['ue_id'][:30], df_srs_v22['gnb_rapl_total_w'][:30], color='#005d5d', marker='x', linestyle=':', label='srsRAN v22.10')
if not df_oai.empty:
    plt.plot(df_oai['ue_id'][:30], df_oai['gnb_rapl_total_w'][:30], color='#fa4d56', marker='s', label='OpenAirInterface (OAI)')

plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('RAPL Package Power (Watts)')
plt.title('Cross-Stack Comparison: Server RAPL Power vs UE Load')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3.3: Comparative - Downlink MCS vs Load
plt.figure(figsize=(9, 5.5))
if not df_srs_v25.empty:
    plt.plot(df_srs_v25['ue_id'][:30], df_srs_v25['ran_dl_mcs'][:30], color='#0f62fe', marker='o', label='srsRAN v25.10 DL MCS')
if not df_srs_v23.empty:
    plt.plot(df_srs_v23['ue_id'][:30], df_srs_v23['ran_dl_mcs'][:30], color='#1192e8', marker='^', linestyle='--', label='srsRAN v23.11 DL MCS')
if not df_srs_v22.empty:
    plt.plot(df_srs_v22['ue_id'][:30], df_srs_v22['ran_dl_mcs'][:30], color='#005d5d', marker='x', linestyle=':', label='srsRAN v22.10 DL MCS')
if not df_oai.empty:
    plt.plot(df_oai['ue_id'][:30], df_oai['ran_dl_mcs'][:30], color='#fa4d56', marker='s', label='OAI DL MCS')

plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('Modulation & Coding Scheme (MCS Index)')
plt.title('Cross-Stack Comparison: DL MCS vs UE Load')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3.4: Comparative - Resident RAM Footprint vs Load
plt.figure(figsize=(9, 5.5))
if not df_srs_v25.empty:
    plt.plot(df_srs_v25['ue_id'][:30], df_srs_v25['ran_proc_rmem_kb'][:30]/1024.0, color='#0f62fe', marker='o', label='srsRAN v25.10')
if not df_srs_v23.empty:
    plt.plot(df_srs_v23['ue_id'][:30], df_srs_v23['ran_proc_rmem_kb'][:30]/1024.0, color='#1192e8', marker='^', linestyle='--', label='srsRAN v23.11')
if not df_srs_v22.empty:
    plt.plot(df_srs_v22['ue_id'][:30], df_srs_v22['ran_proc_rmem_kb'][:30]/1024.0, color='#005d5d', marker='x', linestyle=':', label='srsRAN v22.10')
if not df_oai.empty:
    plt.plot(df_oai['ue_id'][:30], df_oai['ran_proc_rmem_kb'][:30]/1024.0, color='#fa4d56', marker='s', label='OpenAirInterface (OAI)')

plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('Resident Memory (MB)')
plt.title('Cross-Stack Comparison: Process Resident RAM vs UE Load')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3.5: Comparative - Host System Load1 vs Load
plt.figure(figsize=(9, 5.5))
if not df_srs_v25.empty:
    plt.plot(df_srs_v25['ue_id'][:30], df_srs_v25['gnb_load1'][:30], color='#0f62fe', marker='o', label='srsRAN v25.10')
if not df_srs_v23.empty:
    plt.plot(df_srs_v23['ue_id'][:30], df_srs_v23['gnb_load1'][:30], color='#1192e8', marker='^', linestyle='--', label='srsRAN v23.11')
if not df_srs_v22.empty:
    plt.plot(df_srs_v22['ue_id'][:30], df_srs_v22['gnb_load1'][:30], color='#005d5d', marker='x', linestyle=':', label='srsRAN v22.10')
if not df_oai.empty:
    plt.plot(df_oai['ue_id'][:30], df_oai['gnb_load1'][:30], color='#fa4d56', marker='s', label='OpenAirInterface (OAI)')

plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('Linux System Load (Load1)')
plt.title('Cross-Stack Comparison: Host System Load vs UE Load')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3.6: Comparative - Attach Latency vs Load
plt.figure(figsize=(9, 5.5))
if not df_srs_v25.empty:
    plt.plot(df_srs_v25['ue_id'][:30], df_srs_v25['attach_ms'][:30]/1000.0, color='#0f62fe', marker='o', label='srsRAN v25.10')
if not df_srs_v23.empty:
    plt.plot(df_srs_v23['ue_id'][:30], df_srs_v23['attach_ms'][:30]/1000.0, color='#1192e8', marker='^', linestyle='--', label='srsRAN v23.11')
if not df_srs_v22.empty:
    plt.plot(df_srs_v22['ue_id'][:30], df_srs_v22['attach_ms'][:30]/1000.0, color='#005d5d', marker='x', linestyle=':', label='srsRAN v22.10')
if not df_oai.empty:
    plt.plot(df_oai['ue_id'][:30], df_oai['attach_ms'][:30]/1000.0, color='#fa4d56', marker='s', label='OpenAirInterface (OAI)')

plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('Control Plane Attach Duration (s)')
plt.title('Cross-Stack Comparison: UE Attach Latency vs UE Load')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Load Balancing Strategy Comparisons (Unmanaged vs Standard 3GPP vs BPEA-LB)

In [ ]:
# Plot 4.1: Load Balancing Strategies - Per-UE DL Throughput Post-LB
plt.figure(figsize=(9, 5.5))
if not df_lb.empty:
    for strategy, col, marker in [
        ('Unmanaged_Single_gNB', '#da1e28', 'x'),
        ('Standard_3GPP_Early_Handover', '#f1c21b', '^'),
        ('BPEA_LB_Gated_Handover', '#198038', 'o')
    ]:
        sub = df_lb[df_lb['lb_strategy'] == strategy]
        if not sub.empty:
            plt.plot(sub['ue_id'], sub['post_lb_tput_mbps'], label=strategy.replace('_', ' '), color=col, marker=marker)

plt.axhline(y=10.0, color='blue', linestyle=':', label='SLA Target (10 Mbps)')
plt.xlabel('UE ID (Sequential Cell Load)')
plt.ylabel('Post-LB DL Throughput (Mbps)')
plt.title('Load Balancing Comparison: Post-Migration Per-UE Throughput')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 4.2: Load Balancing Strategies - Total Power (gNB1 + gNB2)
plt.figure(figsize=(9, 5.5))
if not df_lb.empty:
    for strategy, col, marker in [
        ('Unmanaged_Single_gNB', '#da1e28', 'x'),
        ('Standard_3GPP_Early_Handover', '#f1c21b', '^'),
        ('BPEA_LB_Gated_Handover', '#198038', 'o')
    ]:
        sub = df_lb[df_lb['lb_strategy'] == strategy]
        if not sub.empty and 'total_system_power_w' in sub.columns:
            plt.plot(sub['ue_id'], sub['total_system_power_w'], label=strategy.replace('_', ' '), color=col, marker=marker)

plt.xlabel('UE ID (Sequential Cell Load)')
plt.ylabel('Total System Power (gNB1 + gNB2 Watts)')
plt.title('Load Balancing Comparison: Total Cluster Power Consumption')
plt.legend()
plt.tight_layout()
plt.show()